# Lezione 13 · Pratica: casi d'uso reali e chiusura del corso

## Il tuo progetto finale

Questo è l'ultimo notebook del corso "Advanced Deep Learning". Non introduce tecniche nuove: ti chiede di combinare, in un mini-progetto di chiusura, componenti che hai già costruito nel corso delle dodici lezioni precedenti. Completi due esercizi che formano una pipeline (flusso di lavoro) completa di classificazione, dall'inizio alla fine, con dati tabulari reali.

- **Esercizio 1.** Costruisci, addestri e valuti un perceptron multistrato (MLP, Multilayer Perceptron) in PyTorch per diagnosticare se un tumore al seno è maligno o benigno, con uno strato di *dropout* incluso nell'architettura.
- **Esercizio 2.** Riutilizzi quel modello per verificare, con i tuoi dati, l'effetto reale della regolarizzazione: confronti l'accuratezza sul test della versione con *dropout* rispetto alla versione senza.

Ogni esercizio include lo scheletro del codice con parti contrassegnate come `# TODO`, che completi tu. Sotto ogni esercizio trovi una soluzione collassata: prova a risolvere l'esercizio da solo prima di aprirla.

## Prerequisito

Devi aver completato il corso "Advanced Deep Learning", in particolare la Lezione 5 (Ottimizzazione I), dove hai visto l'ottimizzatore Adam e lo strato `nn.Dropout` come tecnica di regolarizzazione.

## Come eseguire questo notebook

Aprilo in Google Colab ed esegui le celle in ordine, dall'alto verso il basso. Non installare nulla di aggiuntivo: `torch` e `scikit-learn` sono già preinstallati in Colab. Il dataset viene caricato direttamente da `scikit-learn`, senza download esterni.

## Esercizio 1 · Pipeline completa di diagnosi del cancro al seno

### Il caso

Collabori con un laboratorio clinico che vuole una seconda opinione automatica sulle biopsie di tessuto mammario. Ogni campione arriva già caratterizzato da 30 misure numeriche (raggio, texture, perimetro, simmetria, tra le altre) estratte da un'immagine digitalizzata del tessuto. Il tuo compito è addestrare un modello che, a partire da queste 30 misure, classifichi il campione come maligno o benigno, e che riporti con quale affidabilità lo fa prima che il team clinico valuti di usarlo come supporto.

Usi il dataset "Breast Cancer Wisconsin", integrato in `scikit-learn`: 569 campioni, 30 caratteristiche numeriche per campione e 2 classi.

### Cosa devi fare

Il caricamento dei dati, la suddivisione in addestramento e test, e la normalizzazione sono già scritti e non serve toccarli. Completa due parti:

1. L'architettura del modello nella classe `MLPClasificador`: uno strato nascosto, uno strato `nn.Dropout` (che hai visto nella Lezione 5 come tecnica di regolarizzazione) e lo strato di uscita.
2. Il calcolo delle tre metriche finali sul set di test: precisione (*precision*), richiamo (*recall*) e la metrica F1, usando le funzioni di `sklearn.metrics`.

Il ciclo di addestramento, che usa l'ottimizzatore Adam (Adaptive Moment Estimation), è già scritto: non modificarlo.

### Suggerimenti

- `MLPClasificador` riceve `num_entradas` (30, una per caratteristica), `num_ocultas` (dimensione dello strato nascosto) e `p_dropout` (probabilità di spegnere ogni neurone durante l'addestramento).
- `nn.Dropout(p_dropout)` spegne casualmente una frazione `p_dropout` delle attivazioni a ogni passo di addestramento, e si disattiva automaticamente in modalità di valutazione (`modelo.eval()`).
- Lo strato di uscita produce 2 logit, uno per classe. Non applicare esplicitamente `softmax`: `nn.CrossEntropyLoss` lo include già internamente.
- Non dimenticare di applicare ReLU (unità lineare rettificata) dopo il primo strato lineare.
- Per le metriche, `precision_score(y_verdadero, y_predicho)`, `recall_score(y_verdadero, y_predicho)` e `f1_score(y_verdadero, y_predicho)` si aspettano array di NumPy (o liste) con le etichette reali e quelle previste, in quest'ordine.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score

# Fijamos la semilla para que el resultado sea reproducible
torch.manual_seed(42)
np.random.seed(42)

# Usa GPU (unidad de procesamiento gráfico) si Colab te la ha asignado, si no, usa CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de entrenamiento: {device}")

# Cargamos el dataset de diagnóstico de cáncer de mama: 569 muestras,
# 30 características numéricas por muestra, 2 clases (maligno / benigno)
datos = load_breast_cancer()
X, y = datos.data, datos.target
nombres_clases = datos.target_names
print(f"Clases: {list(nombres_clases)}")

# Separamos en train/test de forma estratificada para conservar
# la proporción de clases en ambos conjuntos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalizamos: media 0, desviación típica 1, ajustado solo con train
# para no filtrar información del conjunto de prueba
escalador = StandardScaler()
X_train = escalador.fit_transform(X_train)
X_test = escalador.transform(X_test)

# Convertimos a tensores de PyTorch
X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.long).to(device)
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_t = torch.tensor(y_test, dtype=torch.long).to(device)

print(f"Muestras de entrenamiento: {X_train_t.shape[0]}, de prueba: {X_test_t.shape[0]}")
print(f"Número de características por muestra: {X_train_t.shape[1]}")

In [ ]:
class MLPClasificador(nn.Module):
    def __init__(self, num_entradas, num_ocultas=32, p_dropout=0.3):
        super().__init__()

        # TODO: define la primera capa lineal.
        # Recibe num_entradas características y produce num_ocultas salidas.
        self.fc1 = ...

        # TODO: define una capa de Dropout con probabilidad p_dropout.
        # Apaga aleatoriamente una fracción de las neuronas durante el
        # entrenamiento para mitigar el sobreajuste (Clase 5).
        self.dropout = ...

        # TODO: define la segunda capa lineal, la de salida.
        # Recibe num_ocultas valores y produce 2 salidas (logits): una por
        # cada clase (maligno / benigno).
        self.fc2 = ...

    def forward(self, x):
        # TODO: aplica fc1 y después la no linealidad ReLU
        x = ...

        # TODO: aplica dropout sobre la salida anterior
        x = ...

        # TODO: aplica fc2 para obtener los logits finales (sin activación)
        x = ...

        return x


modelo = MLPClasificador(num_entradas=X_train_t.shape[1])
modelo = modelo.to(device)
print(modelo)

In [ ]:
criterio = nn.CrossEntropyLoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)

num_epocas = 100
modelo.train()
for epoca in range(1, num_epocas + 1):
    optimizador.zero_grad()
    salidas = modelo(X_train_t)
    perdida = criterio(salidas, y_train_t)
    perdida.backward()
    optimizador.step()

    if epoca % 20 == 0:
        print(f"Época {epoca}/{num_epocas} | pérdida train: {perdida.item():.4f}")

In [ ]:
modelo.eval()
with torch.no_grad():
    salidas_test = modelo(X_test_t)
    predicciones = salidas_test.argmax(dim=1).cpu().numpy()

y_test_np = y_test_t.cpu().numpy()

# TODO: calcula la precisión de las predicciones sobre el conjunto de test
# frente a las etiquetas reales, con precision_score
precision = ...

# TODO: calcula la exhaustividad (recall) con recall_score
exhaustividad = ...

# TODO: calcula la métrica F1 con f1_score
f1 = ...

print(f"Precisión:      {precision:.4f}")
print(f"Exhaustividad:  {exhaustividad:.4f}")
print(f"F1:             {f1:.4f}")

<details>
<summary>💡 Vedi soluzione</summary>

```python
class MLPClasificador(nn.Module):
    def __init__(self, num_entradas, num_ocultas=32, p_dropout=0.3):
        super().__init__()

        # Strato nascosto: da num_entradas caratteristiche a num_ocultas neuroni
        self.fc1 = nn.Linear(num_entradas, num_ocultas)

        # Dropout: spegne casualmente una frazione p_dropout dei
        # neuroni a ogni passo di addestramento (Lezione 5)
        self.dropout = nn.Dropout(p_dropout)

        # Strato di uscita: 2 logit, uno per classe (maligno / benigno)
        self.fc2 = nn.Linear(num_ocultas, 2)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)  # logit, senza softmax esplicito
        return x


modelo = MLPClasificador(num_entradas=X_train_t.shape[1])
modelo = modelo.to(device)
print(modelo)

# --- Calcolo delle metriche finali ---
modelo.eval()
with torch.no_grad():
    salidas_test = modelo(X_test_t)
    predicciones = salidas_test.argmax(dim=1).cpu().numpy()

y_test_np = y_test_t.cpu().numpy()

precision = precision_score(y_test_np, predicciones)
exhaustividad = recall_score(y_test_np, predicciones)
f1 = f1_score(y_test_np, predicciones)

print(f"Precisione:     {precision:.4f}")
print(f"Richiamo:       {exhaustividad:.4f}")
print(f"F1:             {f1:.4f}")
```
</details>

## Esercizio 2 · Verifica l'effetto del *dropout* con i tuoi dati

### Il caso

Il team clinico dell'Esercizio 1 ti fa una domanda molto ragionevole: "Serve davvero quello strato di *dropout*? Quanto cambia il risultato se lo togli?" Nella Lezione 5 hai visto, in teoria, che il *dropout* agisce come tecnica di regolarizzazione e aiuta il modello a generalizzare meglio su dati che non ha mai visto. Ora lo verifichi in modo empirico su questo stesso dataset e con questo stesso modello, invece di darlo per scontato.

### Cosa devi fare

Completa la funzione `comparar_regularizacion(X_train, y_train, X_test, y_test)`. Riutilizza la classe `MLPClasificador` dell'Esercizio 1 (è già disponibile perché hai eseguito quella cella) per addestrare da zero due versioni del modello, a parità di condizioni, e confrontarne l'accuratezza sul set di test:

1. Un modello **con** *dropout* attivo (`p_dropout=0.3`, il valore di default).
2. Un modello **senza** *dropout* (`p_dropout=0.0`, che equivale a disattivarlo).

Alla fine, stampa l'accuratezza di ciascuna versione e una conclusione di una riga su quale generalizza meglio su questo specifico set di test.

### Suggerimenti

- Fissa lo stesso seme (`torch.manual_seed(0)`) subito prima di istanziare ciascun modello, così entrambi partono da pesi iniziali confrontabili e la differenza dipende solo dal *dropout*.
- `nn.Dropout(0.0)` è uno strato valido che non spegne nessun neurone: è un modo semplice per "disattivare" il *dropout* senza cambiare l'architettura.
- Il resto della funzione (il ciclo di addestramento e il calcolo dell'accuratezza sul test) è già risolto: ti manca solo istanziare i due modelli e scrivere il confronto finale.
- Con un dataset di appena 569 campioni e uno strato nascosto piccolo, la differenza tra le due versioni può essere modesta. Anche questo è un risultato valido: non tutti i modelli piccoli sovradattano abbastanza da far sì che il *dropout* faccia una differenza grande.

In [ ]:
def comparar_regularizacion(X_train, y_train, X_test, y_test, num_epocas=100):
    """Entrena dos versiones del mismo MLP -una con dropout activo y otra
    sin dropout- sobre los mismos datos, y compara su exactitud en test."""

    X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_t = torch.tensor(y_train, dtype=torch.long).to(device)
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_test_t = torch.tensor(y_test, dtype=torch.long).to(device)

    torch.manual_seed(0)
    # TODO: instancia un modelo CON dropout.
    # Usa el valor por defecto de MLPClasificador (p_dropout=0.3) y
    # muévelo al device con .to(device)
    modelo_con_dropout = ...

    torch.manual_seed(0)
    # TODO: instancia un modelo SIN dropout.
    # Pasa p_dropout=0.0 y muévelo también al device con .to(device)
    modelo_sin_dropout = ...

    resultados = {}
    versiones = [
        ("con dropout", modelo_con_dropout),
        ("sin dropout", modelo_sin_dropout),
    ]
    for nombre, modelo_actual in versiones:
        criterio = nn.CrossEntropyLoss()
        optimizador = torch.optim.Adam(modelo_actual.parameters(), lr=1e-3)

        modelo_actual.train()
        for _ in range(num_epocas):
            optimizador.zero_grad()
            salidas = modelo_actual(X_train_t)
            perdida = criterio(salidas, y_train_t)
            perdida.backward()
            optimizador.step()

        modelo_actual.eval()
        with torch.no_grad():
            predicciones = modelo_actual(X_test_t).argmax(dim=1).cpu().numpy()
        exactitud = (predicciones == y_test).mean()
        resultados[nombre] = exactitud

    # TODO: imprime la exactitud de cada versión (recorre `resultados`) y
    # una conclusión de una línea indicando cuál de las dos obtuvo mayor
    # exactitud en este conjunto de test
    ...

    return resultados


resultados = comparar_regularizacion(X_train, y_train, X_test, y_test)

<details>
<summary>💡 Vedi soluzione</summary>

```python
def comparar_regularizacion(X_train, y_train, X_test, y_test, num_epocas=100):
    """Addestra due versioni dello stesso MLP — una con dropout attivo e una
    senza dropout — sugli stessi dati, e confronta la loro accuratezza sul test."""

    X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_t = torch.tensor(y_train, dtype=torch.long).to(device)
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_test_t = torch.tensor(y_test, dtype=torch.long).to(device)

    torch.manual_seed(0)
    modelo_con_dropout = MLPClasificador(
        num_entradas=X_train_t.shape[1], p_dropout=0.3
    ).to(device)

    torch.manual_seed(0)
    modelo_sin_dropout = MLPClasificador(
        num_entradas=X_train_t.shape[1], p_dropout=0.0
    ).to(device)

    resultados = {}
    versiones = [
        ("con dropout", modelo_con_dropout),
        ("senza dropout", modelo_sin_dropout),
    ]
    for nombre, modelo_actual in versiones:
        criterio = nn.CrossEntropyLoss()
        optimizador = torch.optim.Adam(modelo_actual.parameters(), lr=1e-3)

        modelo_actual.train()
        for _ in range(num_epocas):
            optimizador.zero_grad()
            salidas = modelo_actual(X_train_t)
            perdida = criterio(salidas, y_train_t)
            perdida.backward()
            optimizador.step()

        modelo_actual.eval()
        with torch.no_grad():
            predicciones = modelo_actual(X_test_t).argmax(dim=1).cpu().numpy()
        exactitud = (predicciones == y_test).mean()
        resultados[nombre] = exactitud

    for nombre, exactitud in resultados.items():
        print(f"Accuratezza {nombre}: {exactitud:.4f}")

    mejor = max(resultados, key=resultados.get)
    print(f"\nIn questo set di test, la versione '{mejor}' generalizza meglio.")

    return resultados


resultados = comparar_regularizacion(X_train, y_train, X_test, y_test)
```
</details>

## Fine del corso: complimenti

Sei arrivato alla fine di "Advanced Deep Learning". Ripercorri la strada che hai fatto:

- Sei partito dalle basi: neuroni artificiali, funzioni di attivazione e l'algoritmo di retropropagazione (*backpropagation*).
- Hai costruito reti neurali convoluzionali (CNN) per la visione artificiale e hai capito architetture più profonde e i loro blocchi residui.
- Hai padroneggiato l'ottimizzazione: discesa del gradiente, le sue varianti, l'ottimizzatore Adam, la regolarizzazione con *dropout* e altre tecniche per evitare il sovradattamento.
- Hai lavorato con le sequenze: reti neurali ricorrenti (RNN) e memoria a lungo e breve termine (LSTM, Long Short-Term Memory).
- Sei entrato nell'elaborazione del linguaggio naturale (NLP) e nei meccanismi di attenzione, fino ad arrivare al Transformer, l'architettura alla base dei modelli linguistici più usati oggi.
- Hai costruito reti generative avversarie (GAN) ed esplorato i modelli generativi moderni e l'apprendimento non supervisionato.
- Hai chiuso con l'intelligenza artificiale (IA) responsabile, e in quest'ultima lezione hai visto come AlphaGo, AlphaFold, la traduzione automatica, WaveNet, le GAN e i modelli linguistici applicano, in produzione, esattamente ciò che tu hai appena costruito in questo mini-progetto: una pipeline completa, con il suo livello di regolarizzazione e la sua valutazione onesta su dati di test.

Ogni esercizio che hai completato, in ciascuna delle tredici lezioni, è un pezzo che ora sai combinare. Questo è il vero risultato del corso: non hai memorizzato formule, hai addestrato il criterio per scegliere l'architettura adeguata davanti a un problema nuovo.

**Continua a costruire i tuoi progetti, applica queste tecniche a un problema reale del tuo settore e condividi ciò che impari con il tuo team: è il modo migliore perché questa conoscenza non resti chiusa in questo notebook.**